<a href="https://colab.research.google.com/github/EvolvingAgentsLabs/agent-forge/blob/main/jit_poc_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.0 MB/s eta 0:00:00


In [2]:
# ===================================================
# CELL 1: Environment Setup
# ===================================================
# @title Step 1: Install Qiskit and Dependencies

# THE FIX: We explicitly install 'qiskit-aer' for the simulator and use
# 'qiskit[visualization]' to correctly pull in matplotlib for plotting.
!pip install -q 'qiskit[visualization]' qiskit-aer numpy

print("✅ Qiskit, Aer Simulator, and all dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Qiskit, Aer Simulator, and all dependencies installed.


In [3]:
import unittest
import numpy as np
import time
from typing import List, Dict, Any, Tuple

# Qiskit for Quantum Simulation
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# ============================================================================
# QUANTUM MEMORY ENGINE - EDUCATIONAL PROOF OF CONCEPT
# ============================================================================
"""
IMPORTANT DISCLAIMER:
This is a simplified educational implementation demonstrating quantum computing
concepts applied to memory systems. Production quantum memory systems would require:
- Quantum error correction (thousands of physical qubits per logical qubit)
- More sophisticated oracle design for complex queries
- Hybrid classical-quantum interfaces for practical scalability
- Solutions to decoherence and noise in NISQ devices

Current Status (2025): This POC runs on classical simulators. Real quantum
hardware with sufficient qubits and coherence time for practical memory
applications is still in development.
"""

class QuantumMemoryEngine:
    """
    An educational prototype demonstrating quantum algorithms for memory operations.

    Key Concepts Demonstrated:
    1. Grover's Search: Provides quadratic speedup (√N vs N) for unstructured search
    2. Entanglement: Shows quantum correlations between related data
    3. Hybrid Approach: Classical pre-filtering + quantum precision search
    """

    def __init__(self):
        self.simulator = AerSimulator()
        self.noise_threshold = 0.01  # Simulated noise for realism
        print("✅ Quantum Memory Engine Initialized (Simulation Mode)")
        print("   Running on: AerSimulator (classical simulation of quantum circuits)")

    def _execute_circuit(self, circuit: QuantumCircuit, shots=1024) -> Dict[str, int]:
        """Execute a quantum circuit and return measurement counts."""
        transpiled_circuit = transpile(circuit, self.simulator)
        result = self.simulator.run(transpiled_circuit, shots=shots).result()
        counts = result.get_counts(transpiled_circuit)

        # Add realistic note about measurement statistics
        if shots < 1000:
            print(f"   Note: Using {shots} shots. Results show statistical distribution.")

        return counts

    def build_grover_search_circuit(self, memories: List[str], query: str) -> Tuple[QuantumCircuit, Dict]:
        """
        Builds a Grover search circuit with proper iteration count.

        Returns:
            circuit: The quantum circuit
            metadata: Dictionary with search details and expected performance
        """
        if not memories:
            raise ValueError("Memory list cannot be empty.")

        n_memories = len(memories)
        n_qubits = int(np.ceil(np.log2(n_memories))) if n_memories > 0 else 1

        # Find matching indices
        matching_indices = [i for i, mem in enumerate(memories) if query.lower() in mem.lower()]

        metadata = {
            'total_memories': n_memories,
            'matching_memories': len(matching_indices),
            'qubits_used': n_qubits,
            'classical_steps': n_memories,  # Worst case for classical
            'quantum_steps': 0,  # Will be calculated
            'speedup_factor': 0  # Will be calculated
        }

        if not matching_indices:
            print(f"⚠️  No match found for '{query}'. Circuit will show uniform distribution.")
            metadata['quantum_steps'] = 0
            qc = QuantumCircuit(n_qubits, n_qubits)
            qc.h(range(n_qubits))
            qc.measure(range(n_qubits), range(n_qubits))
            return qc, metadata

        # Calculate optimal Grover iterations
        # Formula: approximately π/4 * sqrt(N/M) where N=total, M=matches
        optimal_iterations = int(np.pi / 4 * np.sqrt(n_memories / len(matching_indices)))
        optimal_iterations = max(1, optimal_iterations)  # At least 1 iteration

        metadata['quantum_steps'] = optimal_iterations
        metadata['speedup_factor'] = n_memories / (optimal_iterations * 2)  # Approximate speedup

        print(f"📊 Grover's Algorithm Configuration:")
        print(f"   - Searching {n_memories} memories with {n_qubits} qubits")
        print(f"   - Found {len(matching_indices)} matching items")
        print(f"   - Optimal iterations: {optimal_iterations}")
        print(f"   - Theoretical speedup: {metadata['speedup_factor']:.1f}x over classical")

        # Build the circuit
        qc = QuantumCircuit(n_qubits, n_qubits)

        # Initialize superposition
        qc.h(range(n_qubits))
        qc.barrier(label="Init")

        # Grover iterations
        for iteration in range(optimal_iterations):
            # Oracle: Mark the target states
            for index in matching_indices:
                # Convert index to binary
                binary_index = format(index, f'0{n_qubits}b')

                # Add X gates for 0 bits (to make an all-1 state)
                for i, bit in enumerate(reversed(binary_index)):
                    if bit == '0':
                        qc.x(i)

                # Multi-controlled Z gate (phase flip)
                if n_qubits > 1:
                    # Create multi-controlled Z
                    qc.h(n_qubits - 1)
                    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
                    qc.h(n_qubits - 1)
                else:
                    qc.z(0)

                # Undo X gates
                for i, bit in enumerate(reversed(binary_index)):
                    if bit == '0':
                        qc.x(i)

            qc.barrier(label=f"Oracle_{iteration+1}")

            # Diffusion operator (inversion about average)
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))

            # Multi-controlled Z
            qc.h(n_qubits - 1)
            if n_qubits > 1:
                qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
            else:
                qc.z(0)
            qc.h(n_qubits - 1)

            qc.x(range(n_qubits))
            qc.h(range(n_qubits))
            qc.barrier(label=f"Diffusion_{iteration+1}")

        # Measure
        qc.measure(range(n_qubits), range(n_qubits))

        return qc, metadata

    def build_entanglement_circuit(self, memories: Dict[str, str], associations: List[tuple]) -> Tuple[QuantumCircuit, Dict]:
        """
        Creates Bell states between associated memory pairs.

        Returns:
            circuit: The quantum circuit
            metadata: Dictionary with entanglement details
        """
        keys = list(memories.keys())
        n_qubits = len(keys)

        if n_qubits < 2:
            raise ValueError("Need at least 2 memories to demonstrate entanglement")

        metadata = {
            'memory_pairs': len(associations),
            'total_qubits': n_qubits,
            'entangled_pairs': []
        }

        qc = QuantumCircuit(n_qubits, n_qubits)

        print(f"🔗 Creating Entangled Memory States:")

        for key1, key2 in associations:
            if key1 in keys and key2 in keys:
                idx1 = keys.index(key1)
                idx2 = keys.index(key2)

                # Create Bell state between the two qubits
                qc.h(idx1)
                qc.cx(idx1, idx2)

                metadata['entangled_pairs'].append((key1, key2))
                print(f"   - Entangling '{key1}' (Q{idx1}) with '{key2}' (Q{idx2})")

        qc.barrier(label="Entangle")
        qc.measure(range(n_qubits), range(n_qubits))

        return qc, metadata

    def demonstrate_quantum_advantage_threshold(self):
        """
        Shows at what scale quantum advantages become significant.
        """
        print("\n" + "="*60)
        print("   QUANTUM ADVANTAGE SCALING ANALYSIS")
        print("="*60)

        sizes = [8, 100, 1000, 10000, 1000000]

        print("\nDatabase Size | Classical Steps | Quantum Steps | Speedup")
        print("-" * 60)

        for n in sizes:
            classical_steps = n  # Linear search
            quantum_steps = int(np.pi / 4 * np.sqrt(n))  # Grover's
            speedup = classical_steps / quantum_steps

            print(f"{n:>12} | {classical_steps:>15} | {quantum_steps:>13} | {speedup:>7.1f}x")

        print("\n📌 Note: Significant quantum advantage appears at large scales (>1000 items)")
        print("   Current quantum hardware limitations make small-scale demos more feasible.")

# ============================================================================
# UNIT TESTS WITH REALISTIC EXPECTATIONS
# ============================================================================

class TestQuantumMemoryEngine(unittest.TestCase):
    def setUp(self):
        self.engine = QuantumMemoryEngine()

    def test_grover_search_statistical_accuracy(self):
        """Test Grover's search with realistic statistical expectations."""
        print("\n--- Testing Grover's Search with Statistical Distribution ---")

        memories = [
            "Project Phoenix has a deadline of next Friday.",
            "The lead developer is Sarah Chen.",
            "User prefers Python for code examples.",
            "The project database must be PostgreSQL."
        ]
        query = "database"

        circuit, metadata = self.engine.build_grover_search_circuit(memories, query)
        counts = self.engine._execute_circuit(circuit, shots=2048)

        # The target is index 3, which is "11" in binary for 2 qubits
        target_state = "11"

        # Calculate probability
        target_count = counts.get(target_state, 0)
        total_counts = sum(counts.values())
        probability = (target_count / total_counts) * 100

        print(f"\n📊 Results:")
        print(f"   Target state |{target_state}⟩ measured: {target_count}/{total_counts} times")
        print(f"   Success probability: {probability:.1f}%")
        print(f"   Speedup vs classical: {metadata['speedup_factor']:.1f}x")

        # Grover's algorithm should give high probability, but not always 100%
        # Due to discrete rotation angles, we expect 70-100% for small problems
        self.assertGreater(probability, 70,
                          "Grover's algorithm should achieve >70% success probability")

        # Also verify it's the most probable outcome
        most_probable = max(counts, key=counts.get)
        self.assertEqual(most_probable, target_state,
                        f"Target state |{target_state}⟩ should be most probable")

    def test_entanglement_correlation(self):
        """Test quantum entanglement creates correlations."""
        print("\n--- Testing Quantum Entanglement Correlations ---")

        memories = {"project_name": "Phoenix", "lead_dev": "Sarah"}
        associations = [("project_name", "lead_dev")]

        circuit, metadata = self.engine.build_entanglement_circuit(memories, associations)
        counts = self.engine._execute_circuit(circuit, shots=2048)

        # For Bell states, we expect perfect correlation (00 or 11)
        correlated_outcomes = 0
        uncorrelated_outcomes = 0

        for outcome, count in counts.items():
            if outcome[0] == outcome[1]:  # Both 0 or both 1
                correlated_outcomes += count
            else:
                uncorrelated_outcomes += count

        total = correlated_outcomes + uncorrelated_outcomes
        correlation_percentage = (correlated_outcomes / total) * 100

        print(f"\n🔗 Entanglement Results:")
        print(f"   Correlated outcomes (00 or 11): {correlated_outcomes}/{total}")
        print(f"   Correlation strength: {correlation_percentage:.1f}%")
        print(f"   Theoretical expectation: 100% (Bell state)")

        # Bell states should show perfect correlation in ideal conditions
        self.assertGreater(correlation_percentage, 95,
                          "Bell states should show >95% correlation")

    def test_scaling_analysis(self):
        """Verify the quantum advantage scaling claims."""
        print("\n--- Testing Quantum Advantage Scaling ---")

        # Test different memory sizes
        test_sizes = [8, 64, 256]

        for size in test_sizes:
            memories = [f"Memory item {i}" for i in range(size)]
            query = "item 0"  # First item

            circuit, metadata = self.engine.build_grover_search_circuit(memories, query)

            print(f"\n📈 Size={size}: Classical={metadata['classical_steps']} steps, "
                  f"Quantum={metadata['quantum_steps']} steps, "
                  f"Speedup={metadata['speedup_factor']:.1f}x")

            # Verify the speedup calculation is correct
            expected_quantum_steps = int(np.pi / 4 * np.sqrt(size))
            self.assertAlmostEqual(metadata['quantum_steps'], expected_quantum_steps, delta=1)

# ============================================================================
# DEMONSTRATION WITH CONTEXT
# ============================================================================

def demonstrate_quantum_memory_poc():
    """
    Educational demonstration of quantum memory concepts.
    """
    print("\n\n" + "="*70)
    print("    QUANTUM MEMORY ENGINE - EDUCATIONAL DEMONSTRATION")
    print("="*70)
    print("\n⚠️  IMPORTANT: This is a proof-of-concept running on a classical simulator.")
    print("   Real quantum advantages emerge at large scales (>1000 items) with")
    print("   future fault-tolerant quantum computers.\n")

    engine = QuantumMemoryEngine()

    # First, show the scaling advantage
    engine.demonstrate_quantum_advantage_threshold()

    # Demo 1: Grover's Search
    print("\n\n" + "#"*25 + " DEMO 1: GROVER'S SEARCH " + "#"*25)
    print("Demonstrating quadratic speedup for unstructured search\n")

    memory_database = [
        "User name is Alex.",
        "Project is 'Phoenix'.",
        "Deadline is Friday.",
        "Team lead is Sarah Chen.",
        "Language preference is Python.",
        "Database must be PostgreSQL.",  # Target at index 5
        "User ID is alex123.",
        "Meeting is at 9 AM."
    ]

    print(f"📚 Memory Database: {len(memory_database)} items")
    print(f"🔍 Search Query: 'PostgreSQL'\n")

    start_time = time.time()
    grover_circuit, metadata = engine.build_grover_search_circuit(memory_database, "PostgreSQL")
    counts = engine._execute_circuit(grover_circuit, shots=2048)
    end_time = time.time()

    # Analyze results
    most_probable_state = max(counts, key=counts.get)
    recalled_index = int(most_probable_state, 2)
    probability = (counts[most_probable_state] / sum(counts.values())) * 100

    print(f"\n⏱️  Simulation Time: {end_time - start_time:.4f} seconds")
    print(f"📊 Most Probable State: |{most_probable_state}⟩ (index {recalled_index})")
    print(f"💾 Retrieved Memory: '{memory_database[recalled_index]}'")
    print(f"🎯 Success Probability: {probability:.1f}%")
    print(f"⚡ Theoretical Speedup: {metadata['speedup_factor']:.1f}x vs classical\n")

    # Show distribution
    print("Measurement Distribution (2048 shots):")
    sorted_counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True)[:4])
    for state, count in sorted_counts.items():
        idx = int(state, 2)
        prob = (count / sum(counts.values())) * 100
        mem_preview = memory_database[idx][:30] + "..." if idx < len(memory_database) else "Out of range"
        print(f"  |{state}⟩ (idx {idx}): {count:4} times ({prob:5.1f}%) - {mem_preview}")

    # Demo 2: Entanglement
    print("\n\n" + "#"*25 + " DEMO 2: QUANTUM ENTANGLEMENT " + "#"*25)
    print("Demonstrating quantum correlations between related memories\n")

    memories = {
        "project": "Phoenix",
        "deadline": "Friday",
        "priority": "High",
        "status": "Active"
    }

    # Create multiple entanglements
    associations = [
        ("project", "deadline"),
        ("priority", "status")
    ]

    print(f"📦 Memory Items: {list(memories.keys())}")
    print(f"🔗 Creating Entanglements: {associations}\n")

    entanglement_circuit, ent_metadata = engine.build_entanglement_circuit(memories, associations)
    counts = engine._execute_circuit(entanglement_circuit, shots=1024)

    print("\n📊 Measurement Results (showing correlation patterns):")
    print("State | Count | Interpretation")
    print("-" * 50)

    for state, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        prob = (count / sum(counts.values())) * 100
        # Interpret the correlation
        if state[0] == state[1] and state[2] == state[3]:
            correlation = "✓ Both pairs correlated"
        elif state[0] == state[1]:
            correlation = "✓ First pair correlated"
        elif state[2] == state[3]:
            correlation = "✓ Second pair correlated"
        else:
            correlation = "✗ No correlation (noise)"
        print(f"|{state}⟩ | {count:4} ({prob:5.1f}%) | {correlation}")

    print("\n" + "="*70)
    print("💡 KEY INSIGHTS:")
    print("="*70)
    print("1. Grover's algorithm provides √N speedup (quadratic)")
    print("2. Real advantage emerges at scale (>1000 items)")
    print("3. Entanglement enables instant correlation retrieval")
    print("4. Current hardware limitations require hybrid classical-quantum approaches")
    print("5. This POC demonstrates concepts that will be practical with future hardware")
    print("\n🚀 Next Step: Integrate with AI agents for hybrid quantum-classical memory")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == '__main__':
    # Run unit tests first
    print("="*70)
    print("    RUNNING UNIT TESTS - VALIDATING QUANTUM ALGORITHMS")
    print("="*70)

    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestQuantumMemoryEngine)
    runner = unittest.TextTestRunner(verbosity=2)
    test_result = runner.run(suite)

    if test_result.wasSuccessful():
        print("\n✅ All tests passed! Proceeding to demonstration...\n")
        # Run the educational demonstration
        demonstrate_quantum_memory_poc()
    else:
        print("\n❌ Some tests failed. Please review the implementation.")

test_entanglement_correlation (__main__.TestQuantumMemoryEngine.test_entanglement_correlation)
Test quantum entanglement creates correlations. ... 

    RUNNING UNIT TESTS - VALIDATING QUANTUM ALGORITHMS
✅ Quantum Memory Engine Initialized (Simulation Mode)
   Running on: AerSimulator (classical simulation of quantum circuits)

--- Testing Quantum Entanglement Correlations ---
🔗 Creating Entangled Memory States:
   - Entangling 'project_name' (Q0) with 'lead_dev' (Q1)


ok
test_grover_search_statistical_accuracy (__main__.TestQuantumMemoryEngine.test_grover_search_statistical_accuracy)
Test Grover's search with realistic statistical expectations. ... ok
test_scaling_analysis (__main__.TestQuantumMemoryEngine.test_scaling_analysis)
Verify the quantum advantage scaling claims. ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.388s

OK



🔗 Entanglement Results:
   Correlated outcomes (00 or 11): 2048/2048
   Correlation strength: 100.0%
   Theoretical expectation: 100% (Bell state)
✅ Quantum Memory Engine Initialized (Simulation Mode)
   Running on: AerSimulator (classical simulation of quantum circuits)

--- Testing Grover's Search with Statistical Distribution ---
📊 Grover's Algorithm Configuration:
   - Searching 4 memories with 2 qubits
   - Found 1 matching items
   - Optimal iterations: 1
   - Theoretical speedup: 2.0x over classical

📊 Results:
   Target state |11⟩ measured: 2048/2048 times
   Success probability: 100.0%
   Speedup vs classical: 2.0x
✅ Quantum Memory Engine Initialized (Simulation Mode)
   Running on: AerSimulator (classical simulation of quantum circuits)

--- Testing Quantum Advantage Scaling ---
📊 Grover's Algorithm Configuration:
   - Searching 8 memories with 3 qubits
   - Found 1 matching items
   - Optimal iterations: 2
   - Theoretical speedup: 2.0x over classical

📈 Size=8: Classical=8